# Inspect MOJAVE preprocessing

Measurements use `radio_diffusion.data.mojave_metadata` on the full, native-beam CLEAN images before cropping. Selected images and ridge points remain in memory for inspection; no processed files are written.

UV matches are listed alongside the image inputs. The current measurements use the image FITS data and its CLEAN-component table.

In [ ]:
import random

from radio_diffusion.paths import get_path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.visualization import AsinhStretch, ImageNormalize
from IPython.display import display

from radio_diffusion.data.mojave_metadata import (
    JetOrientationSettings,
    crop_spatial_geometry,
    get_noise_jy_per_beam,
    jet_orientation_deg,
    peak_brightness_jy_per_beam,
    total_flux_jy,
    trace_jet_ridgeline,
)
from radio_diffusion.data.preprocess_mojave import PREPROCESSING_SETTINGS, iter_observations
from radio_diffusion.utils import plotting
from radio_diffusion.utils.plotting import ImageOverlay

image_dir = get_path("MOJAVE_FITS_DIR")
uv_dir = get_path("MOJAVE_UV_DIR")

n_examples = 10
random_seed = 42
selected_filenames = []  # Empty selects a reproducible random sample.

noise_method = "four_corner"
orientation_settings = JetOrientationSettings(inner_mas=1.0, outer_mas=3.0, sigma_cut=3.0)
ridgeline_settings = PREPROCESSING_SETTINGS.ridgeline
radial_extent_half_width_deg = 30.0  # On each side of the inner-jet direction.
radial_extent_fade = True  # False uses a dashed arc.
view_radius_mas = 6.5  # None displays the full image; a number sets the plot radius.

In [ ]:
if not image_dir.is_dir():
    raise FileNotFoundError(f"Image directory not found: {image_dir}")
if not isinstance(n_examples, int) or n_examples < 1:
    raise ValueError("n_examples must be a positive integer.")

observations = list(iter_observations(image_dir, uv_dir))
if not observations:
    raise FileNotFoundError(f"No MOJAVE image FITS files found in {image_dir}")

if selected_filenames:
    observations_by_name = {image.name: (image, uv) for image, uv in observations}
    missing = [name for name in selected_filenames if name not in observations_by_name]
    if missing:
        raise FileNotFoundError(f"Requested images not found: {missing}")
    selected_observations = [observations_by_name[name] for name in selected_filenames]
else:
    selected_observations = random.Random(random_seed).sample(
        observations, min(n_examples, len(observations)),
    )

print(f"Images found: {len(observations)}; selected: {len(selected_observations)}")
display(pd.DataFrame([
    {
        "image_filename": image.name,
        "uv_filename": uv.name if uv is not None else None,
    }
    for image, uv in selected_observations
]))

In [ ]:
inspection_examples = []
measurement_rows = []

for image_path, uv_path in selected_observations:
    row = {"image_filename": image_path.name, "error": ""}
    try:
        with fits.open(image_path, memmap=False) as hdul:
            image_hdu = hdul[0]
            clean_components_hdu = hdul["AIPS CC"]
            noise = get_noise_jy_per_beam(image_hdu, noise_method)
            jet_pa = jet_orientation_deg(
                image_hdu, clean_components_hdu, noise, orientation_settings,
            )
            ridge, stopping_reason = trace_jet_ridgeline(
                image_hdu, noise, jet_pa, ridgeline_settings,
            )
            metadata = {
                "source_name": image_path.name.split(".", 1)[0],
                "noise_rms_jy_per_beam": noise,
                "noise_method": noise_method,
                "total_flux_jy": total_flux_jy(clean_components_hdu),
                "peak_brightness_jy_per_beam": peak_brightness_jy_per_beam(image_hdu),
                "jet_pa_deg": jet_pa,
                "jet_radial_extent_mas": ridge[-1]["radius_mas"] if ridge else float("nan"),
                "ridgeline_stop_reason": stopping_reason,
                "ridgeline_arc_clipped": any(point["arc_clipped"] for point in ridge),
            }
            image = np.squeeze(image_hdu.data).copy()
            header = image_hdu.header.copy()
        row.update(metadata)
        inspection_examples.append({
            "image_filename": image_path.name,
            "image": image,
            "header": header,
            "metadata": metadata,
            "ridge": ridge,
            "ridgeline_settings": ridgeline_settings,
        })
    except Exception as error:
        row["error"] = f"{type(error).__name__}: {error}"
    measurement_rows.append(row)

measurement_table = pd.DataFrame(measurement_rows)
with pd.option_context("display.max_colwidth", None):
    display(measurement_table)
print(f"Ready for plotting: {len(inspection_examples)} / {len(selected_observations)} observations")

In [ ]:
def build_example_overlays(example):
    """Prepare the overlays for an individual preview or a grid tile."""
    header = example["header"]
    metadata = example["metadata"]
    ridge = example["ridge"]

    coordinates = plotting.mojave_coordinates(crop_spatial_geometry(header, left=0, top=0))
    overlays = ImageOverlay(coordinates=coordinates)
    overlays.append(ImageOverlay.points(
        [(0, 0)], marker="+", markersize=10, color="white", label="Assumed core",
    ))

    pa_deg = metadata["jet_pa_deg"]
    if np.isfinite(pa_deg):
        direction = np.deg2rad(pa_deg)
        overlays.append(ImageOverlay.arrow(
            (0, 0), (1.5 * np.sin(direction), 1.5 * np.cos(direction)),
            label="Inner-jet direction (1.5 mas arrow)", alpha=0.4,
        ))

    if ridge:
        ridge_pixels = np.array([[point["x_pixel"], point["y_pixel"]] for point in ridge])
        ridge_offsets = coordinates.from_pixels(ridge_pixels)
        if len(ridge_offsets) >= 2:
            overlays.append(ImageOverlay.polyline(
                ridge_offsets, smooth=True, color="deepskyblue",
                linewidth=1, label="Smoothed ridgeline", alpha=0.4,
            ))
        overlays.append(ImageOverlay.points(
            ridge_offsets, color="magenta", label="Raw ridge detections", alpha=0.4
        ))
    extent = metadata["jet_radial_extent_mas"]
    if np.isfinite(pa_deg) and np.isfinite(extent) and extent > 0:
        overlays.append(ImageOverlay.arc(
            (0, 0), extent, angle_deg=90 - pa_deg,
            delta_angle_deg=radial_extent_half_width_deg, color="lime",
            fade=radial_extent_fade, linestyle="solid" if radial_extent_fade else "dashed",
            linewidth=1, label=f"Radial extent: {extent:.2f} mas", alpha=0.4,
        ))

    bmaj, bmin, bpa = (header.get(key) for key in ("BMAJ", "BMIN", "BPA"))
    if (
        all(isinstance(value, (int, float)) and np.isfinite(value) for value in (bmaj, bmin, bpa))
        and 0 < bmin <= bmaj
    ):
        overlays.append(plotting.restoring_beam(bmaj * 3_600_000, bmin * 3_600_000, bpa))

    return overlays


def set_example_view(ax, radius_mas):
    if radius_mas is not None:
        if not np.isfinite(radius_mas) or radius_mas <= 0:
            raise ValueError("Plot radius must be finite and positive, in mas.")
        ax.set(xlim=(radius_mas, -radius_mas), ylim=(-radius_mas, radius_mas))


def show_example(example, view_radius_mas=None):
    """Display the native image, shared measurements and temporary ridge points."""
    image = example["image"]
    metadata = example["metadata"]
    overlays = build_example_overlays(example)
    pa_deg = metadata["jet_pa_deg"]
    peak = metadata["peak_brightness_jy_per_beam"]
    norm = ImageNormalize(
        vmin=0, vmax=max(peak, np.finfo(float).tiny),
        stretch=AsinhStretch(0.01), clip=True,
    )
    fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
    overlays.overlay(image, ax=ax, norm=norm, colorbar=True, colorbar_label="Jy/beam")
    artist = ax.images[-1]
    detection_level = (
        example["ridgeline_settings"].sigma_cut * metadata["noise_rms_jy_per_beam"]
    )
    if np.nanmin(image) < detection_level < peak:
        ax.contour(
            artist.get_array(), levels=[detection_level], colors="white",
            linewidths=0.6, linestyles=":", transform=artist.get_transform(),
        )
        ax.plot([], [], ":", color="white", label="Ridge detection threshold")

    set_example_view(ax, view_radius_mas)

    noise_text = (
        f"Noise ({metadata['noise_method']}): "
        f"{metadata['noise_rms_jy_per_beam']:.3g} Jy/beam"
    )
    status_text = f"Stop: {metadata['ridgeline_stop_reason']}"
    if metadata["ridgeline_arc_clipped"]:
        status_text += "; detected arc window clipped"
    ax.set_title(
        f"{example['image_filename']}\n"
        f"PA: {pa_deg:.2f} deg | {noise_text}\n{status_text}",
        fontsize=10,
    )
    ax.set_facecolor("black")
    ax.legend(loc="upper right", fontsize=7, facecolor="black", labelcolor="white")
    plt.show()
    plt.close(fig)

In [ ]:
for example in inspection_examples:
    show_example(example, view_radius_mas=view_radius_mas)

In [ ]:
if inspection_examples:
    figure, grid_axes = plotting.plot_grid(
        [example["image"] for example in inspection_examples],
        nrow=3,
        sample_labels=[example["image_filename"] for example in inspection_examples],
        overlays=[build_example_overlays(example) for example in inspection_examples],
        colorbar_label="Jy/beam",
    )
    for ax in grid_axes.flat[:len(inspection_examples)]:
        set_example_view(ax, view_radius_mas)
        ax.set_facecolor("black")
        if ax.get_legend() is not None:
            ax.legend(loc="upper right", fontsize=6, facecolor="black", labelcolor="white")
    display(figure)
else:
    print("No inspected images are available for the grid.")

In [ ]:
failed_rows = measurement_table.loc[
    measurement_table["error"].ne(""), ["image_filename", "error"],
]

if failed_rows.empty:
    print("No failed observations in the current selection.")
else:
    with pd.option_context("display.max_colwidth", None):
        display(failed_rows)

    for row in failed_rows.itertuples(index=False):
        print(f"{row.image_filename}: {row.error}")
        try:
            with fits.open(image_dir / row.image_filename, memmap=False) as hdul:
                image = np.squeeze(hdul[0].data)
                if image.ndim != 2 or image.size == 0:
                    raise ValueError(f"Expected a non-empty 2D image; got shape {image.shape}.")
                image = image.copy()
            finite_pixels = image[np.isfinite(image)]
            if finite_pixels.size == 0:
                raise ValueError("Image has no finite pixels.")
        except Exception as error:
            print(f"  Raw image unavailable: {type(error).__name__}: {error}")
            continue

        norm = ImageNormalize(
            vmin=float(finite_pixels.min()), vmax=float(finite_pixels.max()),
            stretch=AsinhStretch(0.01), clip=True,
        )
        fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
        plotting.plot_image(image, ax=ax, norm=norm, colorbar=True, colorbar_label="Native image values")
        ax.invert_yaxis()
        ax.set_title(f"{row.image_filename}\n{row.error}", fontsize=10, wrap=True)
        plt.show()
        plt.close(fig)
